# Function Fitting

## Linear Function


We assume the observed data $\mathbf{y} \in \mathbb{R}^m$ are related to the design matrix $A \in \mathbb{R}^{m \times n}$ by

$$\mathbf{y} = A\,\boldsymbol{\beta} + \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},\, \sigma^2 I)$$

where $\boldsymbol{\beta} \in \mathbb{R}^n$ are the unknown coefficients.  
Each column of $A$ is one **basis function** evaluated at the sample points.

The **least-squares solution** minimises $\|A\boldsymbol{\beta} - \mathbf{y}\|_2^2$.

In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
# Quadratic polynomial:  y = β₀ + β₁ x + β₂ x²
BASIS_LABELS = ["1", "x", "x²"]
TRUE_BETA    = np.array([2.0, -1.5, 0.8])

def build_design(x: np.ndarray) -> np.ndarray:
    """Return design matrix A of shape (len(x), n_basis)."""
    return np.column_stack([np.ones_like(x), x, x**2])

# # Cubic polynomial
# BASIS_LABELS = ["1", "x", "x²", "x³"]
# TRUE_BETA    = np.array([1.0, 0.5, -0.3, 0.1])
# def build_design(x):
#     return np.column_stack([np.ones_like(x), x, x**2, x**3])

# # Sinusoidal basis
# BASIS_LABELS = ["1", "sin(x)", "cos(x)", "sin(2x)"]
# TRUE_BETA    = np.array([1.0, 2.0, -1.0, 0.5])
# def build_design(x):
#     return np.column_stack([np.ones_like(x), np.sin(x), np.cos(x), np.sin(2*x)])

X_RANGE   = (-3.0, 3.0)   # domain
N_SAMPLES = 40            # observations per data set
SIGMA     = 1.0           # true noise standard deviation

n_basis = len(TRUE_BETA)
print(f"Model: y = {' + '.join(f'β{i}·{l}' for i,l in enumerate(BASIS_LABELS))}")
print(f"True β = {TRUE_BETA}")
print(f"n_basis={n_basis}, N={N_SAMPLES}, σ={SIGMA}")

Model: y = β0·1 + β1·x + β2·x²
True β = [ 2.  -1.5  0.8]
n_basis=3, N=40, σ=1.0


## 2  — Generate One Data Set & Fit via SVD

### SVD least-squares

Factorise the design matrix:

$$A = U\,\Sigma\,V^\top, \qquad U \in \mathbb{R}^{m\times n},\; \Sigma = \mathrm{diag}(\sigma_1,\ldots,\sigma_n),\; V \in \mathbb{R}^{n\times n}$$

The least-squares solution is the **pseudoinverse** applied to $\mathbf{y}$:

$$\hat{\boldsymbol{\beta}} = A^+ \mathbf{y} = V\,\Sigma^{-1}\,U^\top\,\mathbf{y}$$

This minimises the residual norm $\|A\hat{\boldsymbol{\beta}} - \mathbf{y}\|_2$.

In [3]:
def sample_data(n=N_SAMPLES, seed=None):
    """Draw x uniformly, evaluate the true function, add Gaussian noise."""
    local_rng = np.random.default_rng(seed)
    x = local_rng.uniform(*X_RANGE, size=n)
    A = build_design(x)
    y = A @ TRUE_BETA + local_rng.normal(0, SIGMA, size=n)
    return x, y, A

def svd_fit(A, y):
    """Fit coefficients via thin SVD. Returns beta_hat, U, s, Vt."""
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    beta_hat = Vt.T @ np.diag(1/s) @ U.T @ y
    return beta_hat, U, s, Vt

# One reference data set
x0, y0, A0 = sample_data(seed=0)
beta_hat, U0, s0, Vt0 = svd_fit(A0, y0)

residuals = y0 - A0 @ beta_hat
dof = N_SAMPLES - n_basis
sigma_hat = np.sqrt(np.sum(residuals**2) / dof)   # unbiased estimate of σ

print("Singular values:", np.round(s0, 3))
print(f"True β    : {TRUE_BETA}")
print(f"Fitted β̂  : {np.round(beta_hat, 4)}")
print(f"σ̂ (from residuals): {sigma_hat:.4f}  (true σ={SIGMA})")

Singular values: [28.659 11.514  4.142]
True β    : [ 2.  -1.5  0.8]
Fitted β̂  : [ 2.1597 -1.4164  0.8372]
σ̂ (from residuals): 1.1102  (true σ=1.0)


In [5]:
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

x_grid = np.linspace(*X_RANGE, 300)
A_grid = build_design(x_grid)
y_true_grid = A_grid @ TRUE_BETA
y_hat_grid  = A_grid @ beta_hat

fig = go.Figure()
fig.add_trace(go.Scatter(x=x0, y=y0, mode="markers",
                         marker=dict(color=COLORS[0], size=6, opacity=0.7),
                         name="Noisy data"))
fig.add_trace(go.Scatter(x=x_grid, y=y_true_grid, mode="lines",
                         line=dict(color=COLORS[1], width=2, dash="dash"),
                         name="True function"))
fig.add_trace(go.Scatter(x=x_grid, y=y_hat_grid, mode="lines",
                         line=dict(color=COLORS[2], width=2),
                         name="SVD fit"))
fig.update_layout(title="Data, True Function & SVD Fit",
                  xaxis_title="x", yaxis_title="y",
                  legend=dict(x=0.02, y=0.98))
fig.show()

## Analytical Error from the SVD

Because $\hat{\boldsymbol{\beta}}$ is a linear function of the noise $\boldsymbol{\varepsilon}$, its covariance is

$$\mathrm{Cov}(\hat{\boldsymbol{\beta}}) = \sigma^2 (A^\top A)^{-1}$$

In the SVD basis $(A^\top A)^{-1} = V\,\Sigma^{-2}\,V^\top$, so

$$\boxed{\mathrm{Cov}(\hat{\boldsymbol{\beta}}) = \sigma^2\,V\,\Sigma^{-2}\,V^\top}$$

The **standard error** of the $i$-th coefficient is therefore $\sqrt{[\mathrm{Cov}]_{ii}}$.  
In practice we substitute the residual estimate $\hat{\sigma}$ for the unknown $\sigma$.

In [6]:
def svd_covariance(s, Vt, sigma):
    """Return the coefficient covariance matrix: σ² V Σ⁻² Vᵀ."""
    V = Vt.T
    return sigma**2 * V @ np.diag(1/s**2) @ V.T

# Using true σ
cov_true  = svd_covariance(s0, Vt0, SIGMA)
se_true   = np.sqrt(np.diag(cov_true))

# Using estimated σ̂
cov_est   = svd_covariance(s0, Vt0, sigma_hat)
se_est    = np.sqrt(np.diag(cov_est))

print(f"{'Basis':>8}  {'True β':>8}  {'β̂':>8}  {'SE (true σ)':>12}  {'SE (σ̂)':>10}")
print("-" * 58)
for i, lbl in enumerate(BASIS_LABELS):
    print(f"{lbl:>8}  {TRUE_BETA[i]:>8.4f}  {beta_hat[i]:>8.4f}"
          f"  {se_true[i]:>12.4f}  {se_est[i]:>10.4f}")

   Basis    True β        β̂   SE (true σ)     SE (σ̂)
----------------------------------------------------------
       1    2.0000    2.1597        0.2377      0.2639
       x   -1.5000   -1.4164        0.0880      0.0977
      x²    0.8000    0.8372        0.0528      0.0587


Generate $K$ **independent** data sets, each with $N$ observations.  
Fit each one with SVD and collect the fitted coefficients $\hat{\boldsymbol{\beta}}^{(k)}$.  
The sample standard deviation over the ensemble estimates the true SE directly from the data-generating process:

$$\hat{\sigma}_{\beta_i}^{\mathrm{ens}} = \mathrm{std}_k\bigl(\hat{\beta}_i^{(k)}\bigr)$$

This is the **gold-standard** empirical reference — it uses extra data and is not available in practice.

In [7]:
N_ENSEMBLE = 2000

ens_betas = np.zeros((N_ENSEMBLE, n_basis))
for k in range(N_ENSEMBLE):
    xk, yk, Ak = sample_data(seed=k+1)
    ens_betas[k], *_ = svd_fit(Ak, yk)

se_ensemble = ens_betas.std(axis=0)
print(f"Ensemble SE ({N_ENSEMBLE} datasets):")
for i, lbl in enumerate(BASIS_LABELS):
    print(f"  β_{i} ({lbl}): {se_ensemble[i]:.4f}")

# Visualise distributions
fig = go.Figure()
for i, lbl in enumerate(BASIS_LABELS):
    fig.add_trace(go.Histogram(x=ens_betas[:, i], nbinsx=60,
                               opacity=0.65, name=f"β_{i} ({lbl})",
                               marker_color=COLORS[i % len(COLORS)]))
    fig.add_vline(x=TRUE_BETA[i], line_dash="dash",
                  line_color=COLORS[i % len(COLORS)])

fig.update_layout(barmode="overlay",
                  title="Ensemble Distribution of Fitted Coefficients",
                  xaxis_title="Coefficient value", yaxis_title="Count")
fig.show()

Ensemble SE (2000 datasets):
  β_0 (1): 0.2436
  β_1 (x): 0.0947
  β_2 (x²): 0.0625


From the **single** reference data set $(x_0, y_0)$, draw $B$ bootstrap resamples (sampling rows with replacement).  
Fit each resample and compute the standard deviation of the bootstrap coefficient estimates:

$$\hat{\sigma}_{\beta_i}^{\mathrm{boot}} = \mathrm{std}_b\bigl(\hat{\beta}_i^{*(b)}\bigr)$$

Bootstrap does not require knowing $\sigma$ or the distributional form — it estimates uncertainty
directly from the **empirical distribution** of the observed data.

In [8]:
N_BOOTSTRAP = 2000

boot_betas = np.zeros((N_BOOTSTRAP, n_basis))
idx_all = np.arange(N_SAMPLES)

for b in range(N_BOOTSTRAP):
    idx = rng.choice(idx_all, size=N_SAMPLES, replace=True)
    boot_betas[b], *_ = svd_fit(A0[idx], y0[idx])

se_bootstrap = boot_betas.std(axis=0)
print(f"Bootstrap SE ({N_BOOTSTRAP} resamples):")
for i, lbl in enumerate(BASIS_LABELS):
    print(f"  β_{i} ({lbl}): {se_bootstrap[i]:.4f}")

NameError: name 'rng' is not defined

In [9]:
methods = ["SVD analytic (true σ)", "SVD analytic (σ̂)", "Ensemble", "Bootstrap"]
se_table = np.vstack([se_true, se_est, se_ensemble, se_bootstrap])  # (4, n_basis)

fig = go.Figure()
for i, lbl in enumerate(BASIS_LABELS):
    fig.add_trace(go.Bar(name=f"β_{i} ({lbl})",
                         x=methods, y=se_table[:, i],
                         marker_color=COLORS[i % len(COLORS)]))

fig.update_layout(barmode="group",
                  title="Standard Error Comparison — All Methods",
                  xaxis_title="Method",
                  yaxis_title="Standard error of coefficient",
                  legend_title="Coefficient")
fig.show()

print(f"\n{'Method':<26}  " + "  ".join(f"SE(β_{i})" for i in range(n_basis)))
print("-" * (26 + 12 * n_basis))
for method, row in zip(methods, se_table):
    vals = "  ".join(f"{v:>10.4f}" for v in row)
    print(f"{method:<26}  {vals}")

NameError: name 'se_bootstrap' is not defined

In [10]:
norm_residuals = residuals / sigma_hat

# Residuals vs x
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=x0, y=norm_residuals, mode="markers",
                          marker=dict(color=COLORS[0], size=6, opacity=0.7),
                          name="Residuals"))
fig1.add_hline(y=0, line_dash="dash", line_color="grey")
fig1.update_layout(title="Normalised Residuals vs x",
                   xaxis_title="x", yaxis_title="Residual / σ̂")
fig1.show()

# Histogram of residuals
z_grid = np.linspace(-4, 4, 200)
gauss = np.exp(-0.5 * z_grid**2) / np.sqrt(2 * np.pi)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(x=norm_residuals, nbinsx=20, histnorm="probability density",
                            marker_color=COLORS[0], opacity=0.7, name="Residuals"))
fig2.add_trace(go.Scatter(x=z_grid, y=gauss, mode="lines",
                          line=dict(color=COLORS[1], width=2),
                          name="N(0,1)"))
fig2.update_layout(title="Residual Distribution vs Standard Normal",
                   xaxis_title="Normalised residual", yaxis_title="Density")
fig2.show()

print(f"\nResidual summary (should be ~N(0,1)):")
print(f"  mean  = {norm_residuals.mean():.4f}  (expect 0)")
print(f"  std   = {norm_residuals.std():.4f}  (expect ~1)")
print(f"  σ̂    = {sigma_hat:.4f}  (true σ = {SIGMA})")


Residual summary (should be ~N(0,1)):
  mean  = -0.0000  (expect 0)
  std   = 0.9618  (expect ~1)
  σ̂    = 1.1102  (true σ = 1.0)


## Non-linear

Given observations $\{(x_i, y_i)\}$ and a model $f(x;\boldsymbol{\theta})$, we minimise the sum of squared residuals:

$$\mathcal{L}(\boldsymbol{\theta}) = \sum_i r_i^2, \qquad r_i = y_i - f(x_i;\boldsymbol{\theta})$$

Unlike the linear case there is no closed-form solution. The **Jacobian** of the residual vector $\mathbf{r}$ with respect to $\boldsymbol{\theta}$ is

$$J_{ij} = \frac{\partial r_i}{\partial \theta_j}$$

and the gradient and approximate Hessian of $\mathcal{L}$ are

$$\nabla\mathcal{L} = -2J^\top\mathbf{r}, \qquad \nabla^2\mathcal{L} \approx 2J^\top J$$

In [11]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

rng = np.random.default_rng(42)


The default is a **damped sinusoid**:

$$f(x;\boldsymbol{\theta}) = A\,e^{-\gamma x}\cos(\omega x + \phi)$$

with parameters $\boldsymbol{\theta} = (A, \gamma, \omega, \phi)$.

In [46]:
PARAM_NAMES = ["A", "γ", "ω", "φ"]
TRUE_THETA  = np.array([3.0, 0.3, 2.0, 0.5])

def model(x: np.ndarray, theta: np.ndarray) -> np.ndarray:
    A, gamma, omega, phi = theta
    return A * np.exp(-gamma * x) * np.cos(omega * x + phi)
THETA_INIT = np.array([2.0, 0.5, 1.8, 0.0])
# PARAM_NAMES = ["A", "α", "B", "β"]
# TRUE_THETA  = np.array([2.0, 0.5, 1.0, 2.0])
# THETA_INIT  = np.array([1.5, 0.3, 0.5, 1.5])
# def model(x, theta):
#     A, alpha, B, beta = theta
#     return A * np.exp(-alpha * x) + B * np.exp(-beta * x)

X_RANGE   = (0.0, 6.0)
N_SAMPLES = 60
SIGMA     = 0.3

x_data = np.sort(rng.uniform(*X_RANGE, size=N_SAMPLES))
y_data = model(x_data, TRUE_THETA) + rng.normal(0, SIGMA, size=N_SAMPLES)

print(f"Function parameters: {dict(zip(PARAM_NAMES, TRUE_THETA))}")
print(f"N={N_SAMPLES}, σ={SIGMA}, x ∈ {X_RANGE}")

Function parameters: {'A': np.float64(3.0), 'γ': np.float64(0.3), 'ω': np.float64(2.0), 'φ': np.float64(0.5)}
N=60, σ=0.3, x ∈ (0.0, 6.0)


In [47]:
x_grid = np.linspace(*X_RANGE, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode="markers",
                         marker=dict(color=COLORS[0], size=6, opacity=0.7),
                         name="Noisy data"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, TRUE_THETA), mode="lines",
                         line=dict(color=COLORS[1], width=2, dash="dash"),
                         name="True function"))
fig.update_layout(title="Data & True Function",
                  xaxis_title="x", yaxis_title="y")
fig.show()

In [48]:

PARAM_NAMES = ["L", "k", "x₀"]
TRUE_THETA  = np.array([5.0, 1.5, 2.0])
THETA_INIT  = np.array([4.0, 1.0, 1.5])
def model(x, theta):
    L, k, x0 = theta
    return L / (1.0 + np.exp(-k * (x - x0)))

X_RANGE   = (0.0, 6.0)
N_SAMPLES = 60
SIGMA     = 0.3

x_data = np.sort(rng.uniform(*X_RANGE, size=N_SAMPLES))
y_data = model(x_data, TRUE_THETA) + rng.normal(0, SIGMA, size=N_SAMPLES)

print(f"Function parameters: {dict(zip(PARAM_NAMES, TRUE_THETA))}")
print(f"N={N_SAMPLES}, σ={SIGMA}, x ∈ {X_RANGE}")

Function parameters: {'L': np.float64(5.0), 'k': np.float64(1.5), 'x₀': np.float64(2.0)}
N=60, σ=0.3, x ∈ (0.0, 6.0)


In [49]:
x_grid = np.linspace(*X_RANGE, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode="markers",
                         marker=dict(color=COLORS[0], size=6, opacity=0.7),
                         name="Noisy data"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, TRUE_THETA), mode="lines",
                         line=dict(color=COLORS[1], width=2, dash="dash"),
                         name="True function"))
fig.update_layout(title="Data & True Function",
                  xaxis_title="x", yaxis_title="y")
fig.show()


LM interpolates between Gauss–Newton, which is fast near the solution and gradient descent, which is better for further awa components via a damping parameter $\lambda$:

$$\delta\boldsymbol{\theta} = \bigl(J^\top J + \lambda\,\mathrm{diag}(J^\top J)\bigr)^{-1} J^\top \mathbf{r}$$

Using $\mathrm{diag}(J^\top J)$ rather than $I$ as the damping matrix makes the step scale-invariant across parameters (Marquardt's improvement over the original Levenberg formulation).

THis is the rule that we are essentially aplying:
- If the proposed step reduces $\mathcal{L}$: accept, set $\lambda \leftarrow \lambda / \nu$
- If the proposed step increases $\mathcal{L}$: reject, set $\lambda \leftarrow \lambda \cdot \nu$

LM is basically self-tuning, so it's automatically becoming more gauss newton like
### Jacobian via finite differences

$$J_{ij} = \frac{\partial r_i}{\partial\theta_j} \approx \frac{f(x_i;\boldsymbol{\theta}+h\mathbf{e}_j) - f(x_i;\boldsymbol{\theta}-h\mathbf{e}_j)}{2h}$$

In [53]:
def residuals(theta, x, y):
    return y - model(x, theta)

def jacobian(theta, x, y, h=1e-6):
    """Central-difference Jacobian of the residual vector, shape (m, n_params)."""
    n = len(theta)
    J = np.zeros((len(x), n))
    for j in range(n):
        dth = np.zeros(n); dth[j] = h
        J[:, j] = (model(x, theta + dth) - model(x, theta - dth)) / (2 * h)
    # Jacobian of residuals r = y - f  →  ∂r/∂θ = -∂f/∂θ
    return -J

def levenberg_marquardt(
    theta0, x, y,
    lam0=1e-2,      # initial damping
    nu=10.0,        # damping scale factor
    max_iter=200,
    tol_grad=1e-8,
    tol_step=1e-10,
):
    theta = theta0.copy().astype(float)
    lam = lam0
    history = []

    r = residuals(theta, x, y)
    cost = r @ r

    for iteration in range(max_iter):
        J = jacobian(theta, x, y)
        JtJ = J.T @ J
        Jtr = J.T @ r
        if np.max(np.abs(Jtr)) < tol_grad:
            print(f"Converged (gradient) at iteration {iteration}")
            break
        D = np.diag(np.diag(JtJ))          # Marquardt scaling
        lhs = JtJ + lam * D
        try:
            delta = np.linalg.solve(lhs, -Jtr)
        except np.linalg.LinAlgError:
            lam *= nu
            continue

        theta_new = theta + delta
        r_new = residuals(theta_new, x, y)
        cost_new = r_new @ r_new

        history.append((cost, lam))

        if cost_new < cost:              
            theta = theta_new
            r = r_new
            cost = cost_new
            lam = max(lam / nu, 1e-14)
        else:                            
            lam = min(lam * nu, 1e12)

        if np.linalg.norm(delta) < tol_step:
            print(f"Converged (step size) at iteration {iteration}")
            break
    else:
        print(f"Reached max iterations ({max_iter})")
    J_final = jacobian(theta, x, y)
    dof = len(x) - len(theta)
    sigma2_hat = cost / dof
    try:
        cov = sigma2_hat * np.linalg.inv(J_final.T @ J_final)
    except np.linalg.LinAlgError:
        cov = np.full((len(theta), len(theta)), np.nan)

    return theta, history, cov

In [54]:
theta_fit, history, cov_fit = levenberg_marquardt(THETA_INIT, x_data, y_data)
se_fit = np.sqrt(np.diag(cov_fit))

print(f"\n{'Param':>6}  {'True':>8}  {'Init':>8}  {'Fit':>10}  {'SE':>10}")
print("-" * 50)
for name, true, init, fit, se in zip(PARAM_NAMES, TRUE_THETA, THETA_INIT, theta_fit, se_fit):
    print(f"{name:>6}  {true:>8.4f}  {init:>8.4f}  {fit:>10.4f}  {se:>10.4f}")

r_final = y_data - model(x_data, theta_fit)
sigma_hat = np.sqrt(np.sum(r_final**2) / (N_SAMPLES - len(THETA_INIT)))
print(f"\nσ̂ = {sigma_hat:.4f}  (true σ = {SIGMA})")

Converged (gradient) at iteration 11

 Param      True      Init         Fit          SE
--------------------------------------------------
     L    5.0000    4.0000      5.0526      0.0662
     k    1.5000    1.0000      1.5157      0.0828
    x₀    2.0000    1.5000      2.0729      0.0410

σ̂ = 0.2707  (true σ = 0.3)


In [55]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode="markers",
                         marker=dict(color=COLORS[0], size=6, opacity=0.7),
                         name="Noisy data"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, TRUE_THETA), mode="lines",
                         line=dict(color=COLORS[1], dash="dash", width=2),
                         name="True function"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, THETA_INIT), mode="lines",
                         line=dict(color=COLORS[3], dash="dot", width=1.5),
                         name="Initial guess"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, theta_fit), mode="lines",
                         line=dict(color=COLORS[2], width=2),
                         name="LM fit"))
fig.update_layout(title="Levenberg–Marquardt Fit",
                  xaxis_title="x", yaxis_title="y")
fig.show()

### Convergence 

In [18]:
costs = [h[0] for h in history]
lams  = [h[1] for h in history]
iters = list(range(len(history)))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Cost  ∑rᵢ²", "Damping parameter  λ"))

fig.add_trace(go.Scatter(x=iters, y=costs, mode="lines",
                         line=dict(color=COLORS[0])), row=1, col=1)
fig.add_trace(go.Scatter(x=iters, y=lams, mode="lines",
                         line=dict(color=COLORS[1])), row=1, col=2)

fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_layout(title="LM Convergence", showlegend=False)
fig.show()

## 4 — Linear Approximation via Polynomial Basis

We now treat the data as coming from an **unknown** function and ask: what degree polynomial fits it best?

For a polynomial of degree $p$ the design matrix is

$$A = \begin{bmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^p \\ \vdots & & & & \vdots \\ 1 & x_m & x_m^2 & \cdots & x_m^p \end{bmatrix}$$

and we fit $\hat{\boldsymbol{\beta}} = A^+\mathbf{y}$ via SVD.

### $k$-fold cross-validation

Split the data into $k$ roughly equal folds.  For each fold $j = 1,\ldots,k$:

1. **Train** on all folds except $j$
2. **Predict** on fold $j$ — these points were never seen during training
3. Record the held-out squared errors

The **CV score** for model complexity $p$ is the mean held-out squared error across all folds:

$$\mathrm{CV}(p) = \frac{1}{m}\sum_{j=1}^{k}\sum_{i\in\text{fold}_j}\bigl(y_i - \hat{f}_j(x_i)\bigr)^2$$

We pick $\hat{p} = \arg\min_p\,\mathrm{CV}(p)$.

In [28]:
MAX_DEGREE = 16    # polynomial degrees to evaluate 0..MAX_DEGREE
K_FOLDS    = 10

def poly_design(x, degree):
    return np.column_stack([x**d for d in range(degree + 1)])

def svd_fit_predict(A_train, y_train, A_test):
    """Fit on training set, predict on test set."""
    beta = np.linalg.lstsq(A_train, y_train, rcond=None)[0]
    return A_test @ beta

def kfold_cv(x, y, degree, k=K_FOLDS, seed=0):
    """Return mean squared CV error for a polynomial of given degree."""
    local_rng = np.random.default_rng(seed)
    idx = local_rng.permutation(len(x))
    folds = np.array_split(idx, k)
    sq_errors = []
    for j in range(k):
        test_idx  = folds[j]
        train_idx = np.concatenate([folds[i] for i in range(k) if i != j])
        A_tr = poly_design(x[train_idx], degree)
        A_te = poly_design(x[test_idx],  degree)
        y_pred = svd_fit_predict(A_tr, y[train_idx], A_te)
        sq_errors.extend((y[test_idx] - y_pred)**2)
    return np.mean(sq_errors)

degrees = np.arange(0, MAX_DEGREE + 1)
cv_scores   = np.array([kfold_cv(x_data, y_data, d) for d in degrees])
train_mse = np.array([
    np.mean((y_data - poly_design(x_data, d) @
             np.linalg.lstsq(poly_design(x_data, d), y_data, rcond=None)[0])**2)
    for d in degrees
])

best_degree = int(degrees[np.argmin(cv_scores)])
print(f"Best polynomial degree (CV): {best_degree}")
print(f"CV score at best degree:    {cv_scores[best_degree]:.4f}")

Best polynomial degree (CV): 7
CV score at best degree:    0.1096


In [20]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=degrees, y=cv_scores, mode="lines+markers",
                         line=dict(color=COLORS[0]), name=f"{K_FOLDS}-fold CV MSE"))
fig.add_trace(go.Scatter(x=degrees, y=train_mse, mode="lines+markers",
                         line=dict(color=COLORS[1], dash="dash"), name="Training MSE"))
fig.add_vline(x=best_degree, line_dash="dot", line_color=COLORS[3],
              annotation_text=f" best = {best_degree}",
              annotation_position="top right")
fig.update_layout(title=f"{K_FOLDS}-Fold Cross-Validation: Polynomial Degree Selection",
                  xaxis_title="Polynomial degree",
                  yaxis_title="Mean squared error",
                  yaxis_type="log")
fig.show()

Next, I tried comparing the nonlinear fit and the cv-optimal polynomial

In [24]:
deg_low  = max(0, best_degree - 3)
deg_best = best_degree
deg_high = min(MAX_DEGREE, best_degree + 4)
def fit_poly(degree):
    A = poly_design(x_data, degree)
    beta = np.linalg.lstsq(A, y_data, rcond=None)[0]
    return poly_design(x_grid, degree) @ beta

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode="markers",
                         marker=dict(color=COLORS[0], size=5, opacity=0.6),
                         name="Data"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, TRUE_THETA), mode="lines",
                         line=dict(color="grey", dash="dash", width=1.5),
                         name="True function"))
fig.add_trace(go.Scatter(x=x_grid, y=model(x_grid, theta_fit), mode="lines",
                         line=dict(color=COLORS[2], width=2),
                         name="LM fit (nonlinear)"))
fig.add_trace(go.Scatter(x=x_grid, y=fit_poly(deg_low), mode="lines",
                         line=dict(color=COLORS[3], width=1.5, dash="dot"),
                         name=f"Poly deg {deg_low} (underfit)"))
fig.add_trace(go.Scatter(x=x_grid, y=fit_poly(deg_best), mode="lines",
                         line=dict(color=COLORS[4], width=2),
                         name=f"Poly deg {deg_best} (CV best)"))
fig.add_trace(go.Scatter(x=x_grid, y=fit_poly(deg_high), mode="lines",
                         line=dict(color=COLORS[5], width=1.5, dash="dot"),
                         name=f"Poly deg {deg_high} (overfit)"))
fig.update_layout(title="Polynomial Fits vs LM Nonlinear Fit",
                  xaxis_title="x", yaxis_title="y")
fig.show()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/74057366.py:8: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/74057366.py:8: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/74057366.py:8: RuntimeWarning:

invalid value encountered in matmul



In [29]:
A_best = poly_design(x_data, deg_best)
beta_best = np.linalg.lstsq(A_best, y_data, rcond=None)[0]
y_poly_pred = A_best @ beta_best
res_lm   = y_data - model(x_data, theta_fit)
res_poly = y_data - y_poly_pred

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("LM residuals",
                                    f"Polynomial deg {deg_best} residuals"))
for col, res, color in [(1, res_lm, COLORS[2]), (2, res_poly, COLORS[4])]:
    fig.add_trace(go.Scatter(x=x_data, y=res, mode="markers",
                             marker=dict(color=color, size=5, opacity=0.7),
                             showlegend=False), row=1, col=col)
    fig.add_hline(y=0, line_dash="dash", line_color="grey", row=1, col=col)

fig.update_yaxes(title_text="Residual", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_xaxes(title_text="x", row=1, col=2)
fig.update_layout(title="Residual Comparison")
fig.show()

rmse_lm   = np.sqrt(np.mean(res_lm**2))
rmse_poly = np.sqrt(np.mean(res_poly**2))
print(f"RMSE — LM nonlinear:       {rmse_lm:.4f}")
print(f"RMSE — Poly deg {deg_best} (CV): {rmse_poly:.4f}")
print(f"True noise σ:              {SIGMA:.4f}")

RMSE — LM nonlinear:       0.3115
RMSE — Poly deg 16 (CV): 0.2475
True noise σ:              0.3000


## Maximum Entropy Models


$$\langle f_i(x) \rangle = \int_{-\infty}^{\infty} p(x)\, f_i(x)\, dx = \mu_i, \qquad i = 1, \ldots, K$$

we have the normalization cosntraint:

$$\int_{-\infty}^{\infty} p(x)\, dx = 1$$

We want to find the $p(x) \ge 0$ that **maximises** the differential entropy

$$S[p] = -\int_{-\infty}^{\infty} p(x) \log p(x)\, dx$$

subject to these $K+1$ linear constraints.

Form the augmented functional (one multiplier $\lambda_0$ for normalisation, one $\lambda_i$ per constraint):

$$\mathcal{F}[p] = -\int p \log p\, dx
  - \lambda_0\!\left(\int p\, dx - 1\right)
  - \sum_{i=1}^{K} \lambda_i\!\left(\int p\, f_i\, dx - \mu_i\right)$$

Setting the **functional derivative** $\delta\mathcal{F}/\delta p(x) = 0$:

$$-\log p(x) - 1 - \lambda_0 - \sum_{i=1}^K \lambda_i f_i(x) = 0$$

Solving for $p(x)$:

$$p^*(x) = \exp\!\left(-1 - \lambda_0 - \sum_{i=1}^K \lambda_i f_i(x)\right) = \frac{1}{Z}\exp\!\left(-\sum_{i=1}^K \lambda_i f_i(x)\right)$$

where the **partition function** $Z = e^{1+\lambda_0}$ is fixed by normalisation.

**Key result:** The MaxEnt distribution always belongs to the **exponential family**.  
The Lagrange multipliers $\{\lambda_i\}$ are determined by substituting $p^*$ back into the $K$ constraint equations — a nonlinear system that must in general be solved numerically.

Then we can find find the Lagrange multipliers

Given a set of constraint functions $\{f_i\}$ and target moments $\{\mu_i\}$, we solve for $\boldsymbol{\lambda}$ by minimising the **dual objective**:

$$g(\boldsymbol{\lambda}) = \log Z(\boldsymbol{\lambda}) - \sum_{i=1}^K \lambda_i \mu_i, \qquad \log Z = \log\int e^{-\sum_i \lambda_i f_i(x)}\, dx$$

The gradient is $\partial g / \partial \lambda_i = \langle f_i \rangle_{p^*} - \mu_i$, which is zero at the solution — confirming the constraints are met.

In [ ]:
# Each entry: (function f_i, target moment μ_i, label)
# Change this list to explore different MaxEnt distributions.
from scipy import optimize, integrate, stats

# Example: constrain ⟨x⟩ and ⟨x²⟩  →  should recover a Gaussian
CONSTRAINTS = [
    (lambda x: x,    0.0,  "⟨x⟩ = 0"),
    (lambda x: x**2, 1.0,  "⟨x²⟩ = 1"),
]

X_LO, X_HI = -6.0, 6.0
N_GRID = 2000
x_grid = np.linspace(X_LO, X_HI, N_GRID)
dx = x_grid[1] - x_grid[0]

fi_vals = np.array([f(x_grid) for f, _, _ in CONSTRAINTS])  
mu_vals = np.array([mu for _, mu, _ in CONSTRAINTS])

def log_Z_and_grad(lam):
    """Log-partition function and its gradient w.r.t. λ."""
    log_p_unnorm = -(fi_vals.T @ lam)

    # Numerical stabilisation: compute log Z with shift correction.
    shift = np.max(log_p_unnorm)
    p_unnorm = np.exp(log_p_unnorm - shift)
    Z_shifted = np.trapz(p_unnorm, x_grid)
    log_Z = float(np.log(Z_shifted) + shift)

    # Normalised density on grid (shift cancels out here).
    p = p_unnorm / Z_shifted
    moments = np.trapz(fi_vals * p, x_grid, axis=1)
    return log_Z, moments

def dual_objective(lam):
    log_z, _ = log_Z_and_grad(lam)
    return float(log_z - lam @ mu_vals)

def dual_gradient(lam):
    _, moments = log_Z_and_grad(lam)
    return np.asarray(moments - mu_vals, dtype=float)

# Solve for λ
lam0 = np.zeros(len(CONSTRAINTS))
result = optimize.minimize(dual_objective, lam0, jac=dual_gradient,
                           method="L-BFGS-B",
                           options={"ftol": 1e-14, "gtol": 1e-10})

lam_opt = result.x

# Recover p*(x)
log_p_unnorm = -(fi_vals.T @ lam_opt)
log_p_unnorm -= log_p_unnorm.max()
p_star = np.exp(log_p_unnorm)
p_star /= np.trapz(p_star, x_grid)

# Verify constraints
print("Lagrange multipliers λ:", np.round(lam_opt, 6))
print("\nConstraint verification:")
entropy = float(np.trapz(p_star * np.log(np.maximum(p_star, 1e-300)), x_grid))
entropy = -entropy
print(f"\nMaxEnt entropy S = {entropy:.6f}")x_grid)
    print(f"  {label:15s}  target={mu:.4f}  achieved={achieved:.6f}")
print(f"  normalisation      achieved={np.trapz(p_star, x_grid):.8f}")

entropy = -np.trapz(p_star * np.log(np.maximum(p_star, 1e-300)), x_grid)
print(f"\nMaxEnt entropy S = {entropy:.6f}")

Lagrange multipliers λ: [ 0. -1.]

Constraint verification:
  ⟨x⟩ = 0          target=0.0000  achieved=-0.000000
  ⟨x²⟩ = 1         target=1.0000  achieved=34.985890
  normalisation      achieved=1.00000000

MaxEnt entropy S = -0.762815


/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:21: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:21: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:21: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:24: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:46: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:46: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/127740287.py:46: RuntimeWarning:

invalid v

In [33]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_grid, y=p_star, mode="lines",
                         line=dict(color=COLORS[0], width=2.5),
                         name="MaxEnt  p*(x)"))

# Overlay the analytic Gaussian for comparison when constraints are ⟨x⟩, ⟨x²⟩
sigma2 = mu_vals[mu_vals > 0][0] if any(mu_vals > 0) else 1.0
mu_mean = mu_vals[0] if len(mu_vals) > 1 else 0.0
gauss = stats.norm.pdf(x_grid, loc=mu_mean, scale=np.sqrt(sigma2))
fig.add_trace(go.Scatter(x=x_grid, y=gauss, mode="lines",
                         line=dict(color=COLORS[1], dash="dash", width=2),
                         name="N(μ, σ²) analytic"))

fig.update_layout(title="Maximum Entropy Distribution",
                  xaxis_title="x", yaxis_title="p(x)")
fig.show()

Suppose we know **only** the second moment about zero:

$$\langle x^2 \rangle = \int_{-\infty}^{\infty} p(x)\, x^2\, dx = \sigma^2$$

plus normalisation. The MaxEnt distribution from Part 1 becomes

$$p^*(x) = \frac{1}{Z} e^{-\lambda x^2}$$

**Finding $\lambda$ and $Z$:**

Normalisation requires

$$Z = \int_{-\infty}^{\infty} e^{-\lambda x^2}\, dx = \sqrt{\frac{\pi}{\lambda}}$$

so $p^*(x) = \sqrt{\lambda/\pi}\, e^{-\lambda x^2}$.  
Enforcing the constraint:

$$\langle x^2 \rangle = \int_{-\infty}^{\infty} x^2 \sqrt{\frac{\lambda}{\pi}} e^{-\lambda x^2}\, dx = \frac{1}{2\lambda} = \sigma^2 \implies \lambda = \frac{1}{2\sigma^2}$$

We can sub this then back in:

$$p^*(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{x^2}{2\sigma^2}\right)$$

The maximum entropy distribution consistent with knowing only the second moment is a zero-mean Gaussian

The MaxEnt entropythen is:

$$S^* = \frac{1}{2}\log(2\pi e\,\sigma^2)$$

In [36]:
sigma2_values = [0.10, 0.5, 1.0, 2.0, 10.0]

fig = go.Figure()
for s2, color in zip(sigma2_values, COLORS):
    # Analytic Gaussian
    p_analytic = stats.norm.pdf(x_grid, 0, np.sqrt(s2))
    S_analytic = 0.5 * np.log(2 * np.pi * np.e * s2)
    fig.add_trace(go.Scatter(x=x_grid, y=p_analytic, mode="lines",
                             line=dict(color=color, width=2),
                             name=f"σ²={s2}, S={S_analytic:.3f}"))

fig.update_layout(title="MaxEnt Distributions: Gaussians for Different vars",
                  xaxis_title="x", yaxis_title="p(x)",
                  legend_title="σ², entropy")
fig.show()

In [38]:
# Show that Gaussian has strictly higher entropy than alternative distributions
# with the same σ² = 1
sigma2_fixed = 1.0

# Distributions to compare (all with variance σ² = 1)
x_pos = x_grid[x_grid > 0]
x_all = x_grid

competitors = {
    "Gaussian (MaxEnt)": stats.norm.pdf(x_all, 0, 1),
    "Laplace (σ²=1)": stats.laplace.pdf(x_all, 0, 1/np.sqrt(2)),
    "Uniform (σ²=1)": stats.uniform.pdf(x_all, -np.sqrt(3), 2*np.sqrt(3)),
    "t(df=5) scaled": stats.t.pdf(x_all * np.sqrt(3/5), df=5) * np.sqrt(3/5),
}

fig = go.Figure()
entropies = {}
for (name, p), color in zip(competitors.items(), COLORS):
    p = np.maximum(p, 1e-300)
    p /= np.trapz(p, x_all)  # renormalise to grid
    S = -np.trapz(p * np.log(p + 1e-300), x_all)
    var = np.trapz(p * x_all**2, x_all)
    entropies[name] = S
    fig.add_trace(go.Scatter(x=x_all, y=p, mode="lines",
                             line=dict(color=color, width=2),
                             name=f"{name}  S={S:.3f}"))

fig.update_layout(title="Entropy Comparison: All Distributions Have var=1",
                  xaxis_title="x", yaxis_title="p(x)",
                  xaxis_range=[-5, 5])
fig.show()

print("Differential entropies (Gaussian is the maximum):")
for name, S in sorted(entropies.items(), key=lambda t: -t[1]):
    marker = "  ← MaxEnt" if "Gaussian" in name else ""
    print(f"  {name:<25}: S = {S:.4f}{marker}")

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/931777134.py:20: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/931777134.py:21: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_83568/931777134.py:22: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



Differential entropies (Gaussian is the maximum):
  t(df=5) scaled           : S = 1.8462
  Gaussian (MaxEnt)        : S = 1.4189  ← MaxEnt
  Laplace (σ²=1)           : S = 1.3446
  Uniform (σ²=1)           : S = 1.2441



### Part 3 — Optimal Estimation of the Gaussian Mean
We know that data $\{x_n\}_{n=1}^N$ are drawn i.i.d. from $\mathcal{N}(\mu, \sigma^2)$ with $\sigma^2$ known and $\mu$ unknown. We want an estimator $\hat\mu(\mathbf{x})$ that is:

- Unbiased: $\mathbb{E}[\hat\mu] = \mu$ for all $\mu$
-Minimum variance: $\mathrm{Var}(\hat\mu)$ is as small as possible

The Cramér–Rao Lower Bound methods:

For any unbiased estimator of $\mu$, the **Cramér–Rao bound** (CRB) states:

$$\mathrm{Var}(\hat\mu) \ge \frac{1}{\mathcal{I}(\mu)}$$

where $\mathcal{I}(\mu)$ is the Fisher information of the full sample:

$$\mathcal{I}(\mu) = -\mathbb{E}\!\left[\frac{\partial^2}{\partial\mu^2} \log p(\mathbf{x};\mu)\right]$$

For $N$ i.i.d. Gaussian observations the log-likelihood is

$$\log p(\mathbf{x};\mu) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{n=1}^N (x_n - \mu)^2$$

Therefore the fisher information ois:
$$\frac{\partial^2 \log p}{\partial \mu^2} = -\frac{N}{\sigma^2} \implies \mathcal{I}(\mu) = \frac{N}{\sigma^2}$$

Therefore the CRB is:

$$\mathrm{Var}(\hat\mu) \ge \frac{\sigma^2}{N}$$

Then we can see the sample mean achieves the bound:

Consider the sample mean $\bar{x} = \frac{1}{N}\sum_{n=1}^N x_n$.

- Unbiased: $\mathbb{E}[\bar{x}] = \mu$
- Variance: $\mathrm{Var}(\bar{x}) = \sigma^2/N$

The sample mean is therefore the mvue and we can therefore say that no unbiased estimator can do better.

We can check the Rao–Blackwell route too:

The result also follows from the Rao–Blackwell theorem. The sufficient statistic for $\mu$ in a Gaussian model is $T = \sum_n x_n$ (it captures all information about $\mu$ in the data). Any unbiased estimator conditioned on $T$ cannot have larger variance, and $\bar{x} = T/N$ is itself a function of $T$, so it is already the best.

More directly: $\bar{x}$ is the maximum likelihood estimator, which for Gaussian models i think should equal the MVUE.

With the weighed estimators:
WE can set a general linear unbiased estimator** $\hat\mu_w = \sum_n w_n x_n$ with $\sum_n w_n = 1$.  
Its variance is $\sigma^2 \sum_n w_n^2$.  
Tehrefore by the Cauchy–Schwarz inequality:

$$\sum_n w_n^2 \ge \frac{1}{N}$$

with equality iff $w_n = 1/N$ for all $n$ — the uniform average.  
No linear unbiased estimator can do better than that! 

In [39]:
TRUE_MU   = 1.5      
TRUE_SIGMA2 = 5.0    
TRUE_SIGMA  = np.sqrt(TRUE_SIGMA2)
N_SAMPLES_LIST = [5, 10, 25, 50, 100, 200]  
N_TRIALS  = 10_000   

print(f"True μ = {TRUE_MU}, σ² = {TRUE_SIGMA2}")
print(f"Cramér–Rao bound σ²/N for N = {N_SAMPLES_LIST}:")
for N in N_SAMPLES_LIST:
    print(f"  N={N:4d}:  CRB = {TRUE_SIGMA2/N:.4f}")

True μ = 1.5, σ² = 5.0
Cramér–Rao bound σ²/N for N = [5, 10, 25, 50, 100, 200]:
  N=   5:  CRB = 1.0000
  N=  10:  CRB = 0.5000
  N=  25:  CRB = 0.2000
  N=  50:  CRB = 0.1000
  N= 100:  CRB = 0.0500
  N= 200:  CRB = 0.0250


In [40]:
def sample_mean(x): return x.mean(axis=1)
def trimmed_mean(x, p=0.1): return stats.trim_mean(x, p, axis=1)  # 10% trimmed
def median_est(x): return np.median(x, axis=1)
def first_obs(x): return x[:, 0]   # just use x_1 — ignores the rest

estimators = {
    "Sample mean  (MVUE)": sample_mean,
    "Trimmed mean (10%)": trimmed_mean,
    "Median": median_est,
    "First observation": first_obs,
}

results = {name: {"bias": [], "var": [], "mse": []} for name in estimators}
crb_vals = [TRUE_SIGMA2 / N for N in N_SAMPLES_LIST]

for N in N_SAMPLES_LIST:
    X = rng.normal(TRUE_MU, TRUE_SIGMA, size=(N_TRIALS, N))
    for name, fn in estimators.items():
        est = fn(X)
        bias = np.mean(est) - TRUE_MU
        var  = np.var(est)
        mse  = np.mean((est - TRUE_MU)**2)
        results[name]["bias"].append(bias)
        results[name]["var"].append(var)
        results[name]["mse"].append(mse)

print(f"{'Estimator':<25}  {'Bias (N=50)':>12}  {'Var (N=50)':>12}  {'CRB (N=50)':>12}")
print("-" * 68)
idx_50 = N_SAMPLES_LIST.index(50)
for name in estimators:
    b = results[name]["bias"][idx_50]
    v = results[name]["var"][idx_50]
    print(f"{name:<25}  {b:>12.5f}  {v:>12.5f}  {TRUE_SIGMA2/50:>12.5f}")

Estimator                   Bias (N=50)    Var (N=50)    CRB (N=50)
--------------------------------------------------------------------
Sample mean  (MVUE)            -0.00274       0.09886       0.10000
Trimmed mean (10%)             -0.00124       0.10518       0.10000
Median                         -0.00174       0.15191       0.10000
First observation               0.01671       5.09255       0.10000


In [41]:
fig = go.Figure()

# Cramér–Rao bound
fig.add_trace(go.Scatter(x=N_SAMPLES_LIST, y=crb_vals, mode="lines",
                         line=dict(color="black", dash="dash", width=2),
                         name="Cramér–Rao bound σ²/N"))

for (name, data), color in zip(results.items(), COLORS):
    fig.add_trace(go.Scatter(x=N_SAMPLES_LIST, y=data["var"],
                             mode="lines+markers",
                             line=dict(color=color, width=2),
                             marker=dict(size=7),
                             name=name))

fig.update_layout(title="Estimator Variance vs Sample Size (Monte Carlo)",
                  xaxis_title="N (sample size)",
                  yaxis_title="Variance of estimator",
                  yaxis_type="log", xaxis_type="log")
fig.show()

In [42]:
# Sampling distributions at fixed N: show sample mean is tightest
N_DEMO = 25
X_demo = rng.normal(TRUE_MU, TRUE_SIGMA, size=(N_TRIALS, N_DEMO))

fig = go.Figure()
x_range_hist = np.linspace(TRUE_MU - 5, TRUE_MU + 5, 300)

for (name, fn), color in zip(estimators.items(), COLORS):
    est_vals = fn(X_demo)
    fig.add_trace(go.Histogram(
        x=est_vals, nbinsx=80, histnorm="probability density",
        opacity=0.55, name=name, marker_color=color
    ))

# Analytic CRB-achieving Gaussian for sample mean
p_opt = stats.norm.pdf(x_range_hist, TRUE_MU, TRUE_SIGMA / np.sqrt(N_DEMO))
fig.add_trace(go.Scatter(x=x_range_hist, y=p_opt, mode="lines",
                         line=dict(color="black", dash="dash", width=2),
                         name=f"N(μ, σ²/{N_DEMO}) — MVUE theory"))
fig.add_vline(x=TRUE_MU, line_dash="dot", line_color="grey")

fig.update_layout(barmode="overlay",
                  title=f"Sampling Distributions of Estimators  (N={N_DEMO})",
                  xaxis_title="Estimated μ", yaxis_title="Density")
fig.show()

In [43]:
# Efficiency: ratio of each estimator's variance to the CRB
# Efficiency = 1 means the estimator achieves the bound
fig = go.Figure()
for (name, data), color in zip(results.items(), COLORS):
    efficiency = [crb / v for crb, v in zip(crb_vals, data["var"])]
    fig.add_trace(go.Scatter(x=N_SAMPLES_LIST, y=efficiency,
                             mode="lines+markers",
                             line=dict(color=color, width=2),
                             marker=dict(size=7),
                             name=name))

fig.add_hline(y=1.0, line_dash="dash", line_color="black",
              annotation_text=" CRB (efficiency = 1)",
              annotation_position="right")
fig.update_layout(title="Statistical Efficiency = CRB / Var(estimator)",
                  xaxis_title="N", yaxis_title="Efficiency",
                  yaxis_range=[0, 1.1])
fig.show()

print("\nEfficiency at N=50 (1.0 = achieves Cramér–Rao bound):")
for name, data in results.items():
    eff = crb_vals[idx_50] / data["var"][idx_50]
    print(f"  {name:<25}: {eff:.4f}")


Efficiency at N=50 (1.0 = achieves Cramér–Rao bound):
  Sample mean  (MVUE)      : 1.0115
  Trimmed mean (10%)       : 0.9508
  Median                   : 0.6583
  First observation        : 0.0196
